In [1]:
import config, utils
import pandas as pd
import json
import models.datacenter
import models.scheduler
import models.job

## Problem setup

### Load jobs from spark traces

In [ ]:
exp_traces_path = f"data/{config.d_i}-{config.d_f}/traces.csv"
traces_df = pd.read_csv(exp_traces_path)

FileNotFoundError: [Errno 2] No such file or directory: 'data/2025-04-15T16:00:00.000Z-2025-04-16T15:00:00.000Z/traces.csv'

In [ ]:

jobs = {
    index : 
    models.job.Job(
        release_time=utils.str_to_date(trace.release_time), 
        release_location=trace.release_location, 
        release_platform=trace.platform.lower(), 
        runtime=trace.runtime_sec,
        n_nodes = trace.n_nodes,
        VM_instance = trace.VM_instance,
    )
    for index, trace in traces_df.iterrows() if index < 30
}

### Load datacenters

In [ ]:
datacenters = {
    index: models.datacenter.Datacenter(
        provider=dc_name.split(".")[0].split("_")[0],
        location=dc_name.split(".")[0].split("_")[1],
        data=json.load(open(config.in_path+dc_name+".json", "r"))
    )
    for index, dc_name in enumerate(config.datacenter_list)
    
}

### Generate VMs

In [ ]:
vm_instance_names= {
    "aws": 
    [
        "c4.large", 
        "m4.large",
        "r4.large",
        "c4.xlarge", 
        "m4.xlarge",
        "r4.xlarge"
    ],
    "gcp": 
    [
        "n2_highcpu-8",
        "n2_standard-8",
        "n2_highmem-8",
        "n2_highmem-4",
        "n2_highcpu-32",
        "n2-standard-4"
    ]
}

In [ ]:
# For this instace we generate 2 VM for each dc. --> 4*2 =8 VMs since we have 4 datacenters and max 8 simultaneous jobs to allocate
vm_instances = {}
i=0
for dc in datacenters.values():
    for vm_name in vm_instance_names[dc.provider][:2]:# + vm_instance_names[dc.provider][:2]:
        vm = dc.add_vm_instance(vm_name = vm_name, n_nodes = 1)
        vm_instances[i]=vm
        i+=1

In [ ]:
print("Problem recap:")
print("Jobs loaded: ", len(jobs))
print("Datacenters loaded: ", len(datacenters))
print("VM instances loaded: ", len(vm_instances))

### Experiments

In [ ]:
from models.objectives import compute_carbon_at_time, compute_water_at_time, compute_land_use_at_time, compute_linear_at_time
objectives = {
    "carbon_opt_1h": compute_carbon_at_time,
    "water_opt_1h": compute_water_at_time,
    "land_use_opt_1h": compute_land_use_at_time,
    "linear_opt_1h": compute_linear_at_time, # 1, 1, 1 -> normalize, in what range? over the max that can be expected
    "geo_baseline": None
}


#### Compute schedule for all algorithms

In [ ]:
import models.orchestrator

footprints = {}
logs = {}

for algo_name, obj in objectives.items():
    if algo_name == "geo_baseline":
        continue
    orchestrator = models.orchestrator.Orchestrator(
        datacenters=datacenters,
        jobs=jobs,
        vm_instances=vm_instances,
        objective=obj
    )
    fp, log = orchestrator.run_simulation()
    footprints[algo_name] = fp
    logs[algo_name] = log

scheduler = models.scheduler.Scheduler(
    datacenters=datacenters,
    jobs=jobs,
    vm_instances=vm_instances
)
footprints["geo_baseline"] = scheduler.geo_baseline_schedule()

In [ ]:
timestamps = utils.get_timestamps(
    init_time=config.dt_i,
    final_time=config.dt_f,
    step_duration=config.step
)

# Results

In [ ]:
from src.plot.plotter import plot_single_algorithm_1h, plot_single_algorithm
for algo_name, footprint in footprints.items():
    if algo_name == "geo_baseline":
        utils.save_output(
            out_path = config.out_path, 
            footprints=footprint, 
            algo_name=algo_name
        )
        plot_single_algorithm(
            footprints = footprint, 
            timestamps = timestamps, 
            out_path=config.out_path, 
            algorithm_name=algo_name
        )
        
    else:
        utils.save_output_1h(
            out_path = config.out_path, 
            footprints=footprint, 
            algo_name=algo_name
            )
        with open(config.out_path+algo_name+"_log.txt", "w") as f:
            f.write(log)

        plot_single_algorithm_1h(
            footprints = footprint, 
            timestamps = timestamps, 
            out_path=config.out_path, 
            algorithm_name=algo_name
        )



In [ ]:
from src.plot.plotter import plot_algorithms_comparison_1h
plot_algorithms_comparison_1h(footprints=footprints, timestamps=timestamps, out_path=config.out_path)